# Parsing TEI files

## 1 Importing needed packages

In [ ]:
# from xml.sax.handler import ContentHandler
# from xml.sax import make_parser
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd
import concurrent.futures

# import time
import re
import os
import subprocess

from utils import (
    # check_xml,
    bkv_nkv_from_verse_id,
    get_data_from_tei,
    check_and_create_file,
)
from constants import DATA_PATH, TMP_PATH

tqdm.pandas()

## 2 Parsing

For parsing BeautifulSoup is used, as it is fast and reliable. 
The TEIFile class is used to work on/extract data from a given TEI file. The class is defined in `TEIFile.py`. 
There is also a `TEIFile_slim.py` class and file which is used in other cases for faster processsing, where not all data is needed. 

## 3 Get data from TEI files

In this notebook we extract all data on manuscript and verses from TEI files. In the other `parse_*.ipynb` notebooks we utilize `JSON` files and `SparQL` queries to dbpedia for further manuscript data.

### 3.1 Get list of all xml files



In [ ]:
# get files
raw_files = sorted(Path("../nt-transcripts").rglob("*.xml"))
# Convert Path objects to strings
raw_files = [str(file) for file in raw_files]

In [ ]:
# Convert Path objects to strings and filter for specific filename patterns
filtered_files = []

for file in raw_files:
    # get file name as string
    file_str = str(file)
    # Check if the file matches either condition
    is_5_digit_xml = re.search(r"/[1234]\d{4}\.xml$", file_str)
    is_nt_grc_file = re.search(r"NT_GRC_", file_str)
    # if either applies add to filtered files
    if is_5_digit_xml or is_nt_grc_file:
        filtered_files.append(file_str)

In [ ]:
raw_files

In [ ]:
filtered_files

### 3.2 Get data from TEI file
#### 3.2.1 Data extraction from multiple TEI files in parallel

As there are many TEI files, it is necessary (for speed) to run the extraction of data in parallel. Use 'max_workers' to set number of cpu cores to be utilised.

In [ ]:
# Create directories if they don't exist
out_dirs = [
    DATA_PATH + "parsed/man",
    DATA_PATH + "parsed/trans",
    DATA_PATH + "parsed/nomsac",
    DATA_PATH + "parsed/errors",
]

for out_dir in out_dirs:
    os.makedirs(out_dir, exist_ok=True)

In [ ]:
# Execute tasks and gather results
with concurrent.futures.ProcessPoolExecutor() as executor:
    # Submit tasks and collect futures
    futures = [
        executor.submit(
            get_data_from_tei,
            file_path,
            clear_only=True,
            verbose=False,
            write_to_file=True,
            trans_out_dir=out_dirs[1],
            man_out_dir=out_dirs[0],
            nomsac_out_dir=out_dirs[2],
            error_out_dir=out_dirs[3],
        )
        # for file_path in well_formed_files
        for file_path in filtered_files
    ]

    # Initialize tqdm progress bar with total number of tasks
    progress_bar = tqdm(total=len(futures), desc="Processing")

    # Gather results
    for future, file_path in zip(
        concurrent.futures.as_completed(futures), filtered_files
    ):
        # Update tqdm progress bar
        progress_bar.update(1)
        # Write currently running file path
        # progress_bar.write(f"Processing {file_path}...")

    # Close the progress bar
    progress_bar.close()

In [ ]:
# # THIS IS JUST FOR TESTING
# out_dirs = [
#     DATA_PATH + "test/parsed/man",
#     DATA_PATH + "test/parsed/trans",
#     DATA_PATH + "test/parsed/nomsac",
#     DATA_PATH + "test/parsed/errors",
# ]

# for out_dir in out_dirs:
#     os.makedirs(out_dir, exist_ok=True)

# filtered_files = [
#     "../nt-transcripts/ntvmr/10001.xml",
#     "../nt-transcripts/igntp/ecm_1corinthians/NT_GRC_1_1Cor.xml",
# ]

# for file_path in filtered_files:
#     get_data_from_tei(
#         file_path,
#         clear_only=True,
#         verbose=False,
#         write_to_file=True,
#         trans_out_dir=out_dirs[1],
#         man_out_dir=out_dirs[0],
#         nomsac_out_dir=out_dirs[2],
#         error_out_dir=out_dirs[3],
#     )

## 3.2.2 Merge data into one file respectively and re-read into dataframes

In [ ]:
# Concatenating like this is done, as we know all files do have the same header.
# Also, this is computationally more efficient than first reading each file into a pd.DataFrame and then merging those into one.
print("Merging files...")

commands = [
    f"awk 'NR == 1 || FNR > 1' {DATA_PATH}parsed/man/*.csv > {DATA_PATH}parsed/manuscripts.csv",
    f"awk 'NR == 1 || FNR > 1' {DATA_PATH}parsed/trans/*.csv > {DATA_PATH}parsed/verses.csv",
    f"awk 'NR == 1 || FNR > 1' {DATA_PATH}parsed/nomsac/*.csv > {DATA_PATH}parsed/nomsacs.csv",
    f"jq -s '.' {DATA_PATH}parsed/errors/*.json > {DATA_PATH}parsed/errors.json",
]

# Run the commands
for command in commands:
    subprocess.run(command, shell=True, check=True)

In [ ]:
verses_df = pd.read_csv(
    DATA_PATH + "parsed/verses.csv",
    dtype={
        "lection": "string",
        "verse": "string",
        "witness": "string",
        "transcript": "string",
        "text": "string",
        "publisher": "string",
        "publisher": "string",
        "source": "string",
        "ga": "string",
        "sponsor": "string",
        "funder": "string",
        "edition_version": "float",
        "edition_date": "string",
        "publishing_date": "string",
        "encoding_version": "float",
    },
)
manuscripts_df = pd.read_csv(
    DATA_PATH + "parsed/manuscripts.csv",
    dtype={"ga": "string", "docID": "string", "label": "string", "source": "string"},
)
nomsac_df = pd.read_csv(
    DATA_PATH + "parsed/nomsacs.csv",
    dtype={"ga": "string", "lection": "string", "verse": "string", "nomsac": "string"},
)

#### 3.2.3 Apply cleanup and aggregation functions to dataframes

In [ ]:
# merge verses into nomsac by ga, lection, verse
merged_df = pd.merge(nomsac_df, verses_df, on=["ga", "lection", "verse"], how="left")
# drop where no verse identifier is given
merged_df.dropna(subset=["verse"], inplace=True)
# drop unnecessary columns
merged_df.drop(
    columns=[
        "publisher",
        "sponsor",
        "funder",
        "edition_version",
        "edition_date",
        "publishing_date",
        "encoding_version",
    ],
    inplace=True,
)

nomsac_df = merged_df.fillna("NA")

In [ ]:
# Apply the conversion function to the 'bkv' column in the DataFrame
verses_df = verses_df.progress_apply(bkv_nkv_from_verse_id, axis=1)

In [ ]:
verses_df.drop(columns=["verse"], inplace=True)

In [ ]:
verses_df.dropna(subset=["transcript"], inplace=True)

In [ ]:
# manuscripts_df["docID"] = manuscripts_df.progress_apply(ga_to_docID, axis=1)

In [ ]:
verses_df = verses_df.fillna("NA")

### 3.3 Save data frames to CSV files


In [ ]:
print("Write to CSV")

In [ ]:
# sort by GA then by BKV
verses_df.sort_values(by=["ga", "bkv"], inplace=True)
# write to file
check_and_create_file(TMP_PATH + "verses.csv")
verses_df.to_csv(TMP_PATH + "verses.csv", index=False, index_label="index")
# verses_df.to_parquet(TMP_PATH + "verses.parquet", index=False)

In [ ]:
manuscripts_df.sort_values(by="ga", inplace=True)
manuscripts_df.drop_duplicates(inplace=True)
check_and_create_file(TMP_PATH + "manuscripts_tei.csv")
manuscripts_df.to_csv(
    TMP_PATH + "manuscripts_tei.csv", index=False, index_label="index"
)
# verses_df.to_parquet(TMP_PATH + "manuscripts_tei.parquet", index=False)

In [ ]:
nomsac_df.drop_duplicates(inplace=True)
check_and_create_file(TMP_PATH + "nomsacs.csv")
nomsac_df.to_csv(TMP_PATH + "nomsacs.csv", index=False, index_label="index")